# Guia Prático Completo - Ciência de Dados

Notebook de revisão do básico ao projeto integrador.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, r2_score


## 1. Carregando a base


In [ ]:
df = pd.read_csv('../dados/base_exemplo.csv')
df.head()


## 2. Inspeção


In [ ]:
print(df.info())
display(df.describe())
print(df.columns)


## 3. LGPD - minimização


In [ ]:
df_seguro = df.drop(columns=['Email'])
df_seguro.head()


## 4. Estatística descritiva


In [ ]:
print('Moda da região:', df_seguro['Regiao'].mode()[0])
print('Média:', df_seguro['Gasto_Site_R$'].mean())
print('Mediana:', df_seguro['Gasto_Site_R$'].median())
print('Desvio padrão:', df_seguro['Gasto_Site_R$'].std())


## 5. Quartis e IQR


In [ ]:
q1 = df_seguro['Gasto_Site_R$'].quantile(0.25)
q3 = df_seguro['Gasto_Site_R$'].quantile(0.75)
iqr = q3 - q1

li = q1 - 1.5 * iqr
ls = q3 + 1.5 * iqr

print(q1, q3, iqr, li, ls)

df_iqr = df_seguro[
    (df_seguro['Gasto_Site_R$'] >= li) &
    (df_seguro['Gasto_Site_R$'] <= ls)
]

print('Removidos:', len(df_seguro) - len(df_iqr))


## 6. Gráficos


In [ ]:
sns.boxplot(data=df_seguro, y='Gasto_Site_R$')
plt.show()

sns.barplot(data=df_seguro, x='Regiao', y='Gasto_Site_R$')
plt.show()

sns.scatterplot(
    data=df_seguro,
    x='Tempo_Tela_Min',
    y='Gasto_Site_R$',
    alpha=0.3
)
plt.show()


## 7. Teste A/B


In [ ]:
A = df_seguro[df_seguro['Versao_Site'] == 'A']['Tempo_Tela_Min']
B = df_seguro[df_seguro['Versao_Site'] == 'B']['Tempo_Tela_Min']

t, p = stats.ttest_ind(A, B)

print('t:', t)
print('p:', p)

if p < 0.05:
    print('Rejeita H0')
else:
    print('Não rejeita H0')


## 8. Correlação


In [ ]:
r = df_seguro['Tempo_Tela_Min'].corr(
    df_seguro['Gasto_Site_R$']
)
print('Pearson:', r)

matriz = df_seguro.corr(numeric_only=True)
sns.heatmap(matriz, annot=True, cmap='coolwarm', fmt='.2f')
plt.show()


## 9. Regressão linear


In [ ]:
df_modelo = df_seguro[df_seguro['Gasto_Site_R$'] < 5000]

X = df_modelo[['Tempo_Tela_Min']]
y = df_modelo['Gasto_Site_R$']

modelo = LinearRegression()
modelo.fit(X, y)

pred = modelo.predict(X)

r2 = r2_score(y, pred)
rmse = root_mean_squared_error(y, pred)

print('coef:', modelo.coef_[0])
print('intercepto:', modelo.intercept_)
print('R²:', r2)
print('RMSE:', rmse)


## 10. Previsão para 45 minutos


In [ ]:
previsao = modelo.predict([[45]])[0]
piso = previsao - rmse
teto = previsao + rmse

print(f'Piso: R$ {piso:.2f}')
print(f'Previsão: R$ {previsao:.2f}')
print(f'Teto: R$ {teto:.2f}')
